In [13]:
"""
QAOA Classical Preprocessing for MaxCut
========================================
Implements the classical preprocessing described in Farhi, Goldstone, Gutmann (2014).

Key idea: For fixed p, F_p(γ,β) = Σ_{<jk>} <s| U†(C,γ)...U†(B,β) C_{jk} U(B,β)...U(C,γ) |s>
Each edge's contribution only depends on its p-hop neighborhood subgraph.
We simulate these small subgraphs exactly using numpy/scipy, then optimize
the angles classically before ever touching a quantum computer.
"""

import numpy as np
from scipy.optimize import minimize, differential_evolution
from scipy.linalg import expm
from itertools import product
import networkx as nx
from functools import lru_cache


# ─────────────────────────────────────────────
#  Pauli matrices and tensor product helpers
# ─────────────────────────────────────────────

I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Z  = np.array([[1, 0], [0, -1]], dtype=complex)


def kron_n(*ops):
    """Tensor product of a list of matrices."""
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result


def single_qubit_op(op, qubit, n_qubits):
    """Embed a single-qubit operator into an n-qubit Hilbert space."""
    ops = [I2] * n_qubits
    ops[qubit] = op
    return kron_n(*ops)


def two_qubit_zz(j, k, n_qubits):
    """σ_z^j ⊗ σ_z^k embedded in n-qubit space."""
    ops = [I2] * n_qubits
    ops[j] = Z
    ops[k] = Z
    return kron_n(*ops)


# ─────────────────────────────────────────────
#  Build subgraph Hamiltonians
# ─────────────────────────────────────────────

def build_C_maxcut(subgraph, node_list):
    """
    Build the MaxCut objective operator C_G for a subgraph.
    C_{<jk>} = (1 - σ_z^j σ_z^k) / 2  for each edge.
    """
    n = len(node_list)
    node_index = {v: i for i, v in enumerate(node_list)}
    dim = 2 ** n
    C = np.zeros((dim, dim), dtype=complex)
    for u, v in subgraph.edges():
        if u in node_index and v in node_index:
            j, k = node_index[u], node_index[v]
            C += 0.5 * (np.eye(dim) - two_qubit_zz(j, k, n))
    return C


def build_B(n_qubits):
    """B = Σ_j σ_x^j (the mixer Hamiltonian)."""
    dim = 2 ** n_qubits
    B = np.zeros((dim, dim), dtype=complex)
    for j in range(n_qubits):
        B += single_qubit_op(X, j, n_qubits)
    return B


def build_C_edge(j, k, n_qubits):
    """C_{<jk>} = (1 - σ_z^j σ_z^k) / 2 for a single edge."""
    dim = 2 ** n_qubits
    return 0.5 * (np.eye(dim) - two_qubit_zz(j, k, n_qubits))


def initial_state(n_qubits):
    """Uniform superposition |s> = (1/√2^n) Σ_z |z>  (= |+>^⊗n)."""
    dim = 2 ** n_qubits
    return np.ones(dim, dtype=complex) / np.sqrt(dim)


# ─────────────────────────────────────────────
#  Core QAOA simulation on a subgraph
# ─────────────────────────────────────────────

def f_subgraph(subgraph, edge, gammas, betas):
    """
    Compute the contribution of one edge to F_p(γ,β).
    This is equation (24) in the paper:
        f_g(γ,β) = <s,g| U†(C_g,γ_1)...U†(B_g,β_p) C_{jk} U(B_g,β_p)...U(C_g,γ_1) |s,g>

    Parameters
    ----------
    subgraph : nx.Graph  — the p-hop neighborhood subgraph
    edge     : (u,v)     — the central edge whose contribution we compute
    gammas   : array of length p
    betas    : array of length p
    """
    node_list = list(subgraph.nodes())
    n = len(node_list)
    node_index = {v: i for i, v in enumerate(node_list)}
    p = len(gammas)

    C_full = build_C_maxcut(subgraph, node_list)
    B_full = build_B(n)

    # Central edge operator (only this one is measured)
    j_idx = node_index[edge[0]]
    k_idx = node_index[edge[1]]
    C_jk  = build_C_edge(j_idx, k_idx, n)

    # Build state: U(B,β_p) U(C,γ_p) ... U(B,β_1) U(C,γ_1) |s>
    state = initial_state(n)
    for layer in range(p):
        UC = expm(-1j * gammas[layer] * C_full)
        UB = expm(-1j * betas[layer]  * B_full)
        state = UC @ state
        state = UB @ state

    # Expectation <state| C_{jk} |state>
    return np.real(state.conj() @ (C_jk @ state))


# ─────────────────────────────────────────────
#  p-hop neighborhood extraction
# ─────────────────────────────────────────────

def p_hop_subgraph(G, edge, p):
    """
    Return the subgraph induced by all nodes within graph distance p
    from either endpoint of `edge`.
    """
    u, v = edge
    reachable = set()
    for start in [u, v]:
        lengths = nx.single_source_shortest_path_length(G, start, cutoff=p)
        reachable |= set(lengths.keys())
    return G.subgraph(reachable).copy()


# ─────────────────────────────────────────────
#  F_p(γ,β) — the full expectation value
# ─────────────────────────────────────────────

def compute_Fp(G, gammas, betas, p):
    """
    Compute F_p(γ,β) = Σ_{<jk>} f_{g(j,k)}(γ,β)
    
    For bounded-degree graphs, isomorphic subgraphs give identical f values,
    so we cache by graph isomorphism class (approximated here by edge-sorted
    adjacency for small subgraphs).
    """
    total = 0.0
    for edge in G.edges():
        subg = p_hop_subgraph(G, edge, p)
        contrib = f_subgraph(subg, edge, gammas, betas)
        total += contrib
    return total


# ─────────────────────────────────────────────
#  Angle optimization
# ─────────────────────────────────────────────

def optimize_angles(G, p, method="COBYLA", n_restarts=5, verbose=True):
    """
    Find (γ*, β*) = argmax F_p(γ,β).

    Returns
    -------
    best_angles : dict with keys 'gammas', 'betas'
    best_Fp     : float, the maximum F_p value found
    """
    def neg_Fp(params):
        gammas = params[:p]
        betas  = params[p:]
        return -compute_Fp(G, gammas, betas, p)

    best_val = -np.inf
    best_params = None

    for trial in range(n_restarts):
        # Random initialization in [0, 2π]^p × [0, π]^p
        g0 = np.random.uniform(0, 2 * np.pi, size=p)
        b0 = np.random.uniform(0, np.pi,     size=p)
        x0 = np.concatenate([g0, b0])

        bounds = [(0, 2*np.pi)] * p + [(0, np.pi)] * p

        if method == "differential_evolution":
            result = differential_evolution(neg_Fp, bounds, seed=trial,
                                            maxiter=1000, tol=1e-8,
                                            workers=1, updating='deferred')
        else:
            result = minimize(neg_Fp, x0, method=method,
                              options={"maxiter": 10000, "rhobeg": 0.5})

        if -result.fun > best_val:
            best_val    = -result.fun
            best_params = result.x

        if verbose:
            print(f"  restart {trial+1}/{n_restarts}: F_p = {-result.fun:.6f}")

    gammas = best_params[:p]
    betas  = best_params[p:]
    return {"gammas": gammas, "betas": betas}, best_val


# ─────────────────────────────────────────────
#  Approximation ratio estimate
# ─────────────────────────────────────────────

def maxcut_upper_bound(G):
    """Simple upper bound: number of edges (every edge satisfied)."""
    return G.number_of_edges()


def approximation_ratio(Fp, G):
    return Fp / maxcut_upper_bound(G)


# ─────────────────────────────────────────────
#  Demo
# ─────────────────────────────────────────────

if __name__ == "__main__":
    np.random.seed(42)

    print("=" * 60)
    print("QAOA Classical Preprocessing Demo")
    print("=" * 60)

    # ── Example 1: Ring graph (2-regular), p=1,2,3
    # Paper proves M_p = n(2p+1)/(2p+2) analytically
    print("\n[1] Ring of Disagrees — 2-regular graph, n=8")
    n_ring = 8
    G_ring = nx.cycle_graph(n_ring)
    for p in [1]:
        print(f"\n  p = {p}")
        angles, Fp = optimize_angles(G_ring, p=p, n_restarts=3, verbose=True)
        theory     = n_ring * (2*p + 1) / (2*p + 2)
        print(f"  Optimal F_p  = {Fp:.6f}")
        print(f"  Theory (paper): {theory:.6f}")
        print(f"  Approx ratio  = {approximation_ratio(Fp, G_ring):.4f}")
        print(f"  γ* = {np.round(angles['gammas'], 4)}")
        print(f"  β* = {np.round(angles['betas'],  4)}")

    # ── Example 2: 3-regular graph, p=1
    # Paper proves worst-case approximation ratio ≥ 0.6924
    print("\n" + "=" * 60)
    print("[2] 3-regular graph (Petersen graph), p=1")
    G3 = nx.petersen_graph()   # 10 nodes, 3-regular
    print(f"  Nodes: {G3.number_of_nodes()}, Edges: {G3.number_of_edges()}")
    angles, Fp = optimize_angles(G3, p=1, n_restarts=5, verbose=True)
    print(f"\n  Optimal F_1  = {Fp:.6f}")
    print(f"  Upper bound  = {maxcut_upper_bound(G3)}")
    print(f"  Approx ratio = {approximation_ratio(Fp, G3):.4f}  (paper guarantees ≥ 0.6924)")
    print(f"  γ* = {np.round(angles['gammas'], 4)}")
    print(f"  β* = {np.round(angles['betas'],  4)}")

    # ── Example 3: Random 3-regular graph
    print("\n" + "=" * 60)
    print("[3] Random 3-regular graph, n=12, p=1")
    G_rand = nx.random_regular_graph(3, 12, seed=7)
    angles, Fp = optimize_angles(G_rand, p=1, n_restarts=5, verbose=True)
    print(f"\n  Optimal F_1  = {Fp:.6f}")
    print(f"  Upper bound  = {maxcut_upper_bound(G_rand)}")
    print(f"  Approx ratio = {approximation_ratio(Fp, G_rand):.4f}")
    print(f"  γ* = {np.round(angles['gammas'], 4)}")
    print(f"  β* = {np.round(angles['betas'],  4)}")

QAOA Classical Preprocessing Demo

[1] Ring of Disagrees — 2-regular graph, n=8

  p = 1
  restart 1/3: F_p = 6.000000
  restart 2/3: F_p = 6.000000
  restart 3/3: F_p = 6.000000
  Optimal F_p  = 6.000000
  Theory (paper): 6.000000
  Approx ratio  = 0.7500
  γ* = [2.3563]
  β* = [2.7488]

[2] 3-regular graph (Petersen graph), p=1
  Nodes: 10, Edges: 15
  restart 1/5: F_p = 10.386751
  restart 2/5: F_p = 10.386751
  restart 3/5: F_p = 10.386751
  restart 4/5: F_p = 10.386751
  restart 5/5: F_p = 10.386751

  Optimal F_1  = 10.386751
  Upper bound  = 15
  Approx ratio = 0.6925  (paper guarantees ≥ 0.6924)
  γ* = [5.6677]
  β* = [1.1781]

[3] Random 3-regular graph, n=12, p=1
  restart 1/5: F_p = 12.464102
  restart 2/5: F_p = 12.464102
  restart 3/5: F_p = 12.464102
  restart 4/5: F_p = 12.464102
  restart 5/5: F_p = 12.464102

  Optimal F_1  = 12.464102
  Upper bound  = 18
  Approx ratio = 0.6925
  γ* = [2.5262]
  β* = [1.9635]


In [15]:
import rustworkx as rx
import networkx as nx
import numpy as np

n = 20

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n, 1))

edge_list = [
    # Bottom cluster (0, 1, 2, 3, 4, 5)
    (0, 1, 1.0), (1, 2, 1.0),
    (3, 4, 1.0), (4, 5, 1.0),
    (0, 3, 1.0), (1, 4, 1.0), (2, 5, 1.0),
    (0, 4, 1.0), (1, 3, 1.0), # Diagonals for left square
    (1, 5, 1.0), (2, 4, 1.0), # Diagonals for right square

    # Top cluster (11, 12, 13, 14, 15, 16)
    (16, 15, 1.0), (15, 11, 1.0),
    (14, 13, 1.0), (13, 12, 1.0),
    (16, 14, 1.0), (15, 13, 1.0), (11, 12, 1.0),
    (16, 13, 1.0), (14, 15, 1.0), # Diagonals for left square
    (15, 12, 1.0), (11, 13, 1.0), # Diagonals for right square

    # Left triangle cluster (17, 18, 19)
    (17, 18, 1.0),
    (17, 19, 1.0),
    (18, 19, 1.0),

    # Right square/branch cluster (6, 7, 8, 9, 10)
    (8, 9, 1.0),
    (6, 7, 1.0),
    (8, 6, 1.0),
    (9, 7, 1.0),
    (10, 9, 1.0),

    # Bridging edges between clusters
    (4, 13, 1.0),   # Center vertical bridge
    (18, 14, 1.0),  # Top-left bridge
    (19, 3, 1.0),   # Bottom-left bridge
    (12, 8, 1.0),   # Top-right bridge to square
    (5, 6, 1.0),    # Bottom-right bridge to square
    (11, 10, 1.0)   # Top-right bridge to node 10
]
graph.add_edges_from(edge_list)

# Convert rustworkx → networkx
G = nx.Graph()
G.add_nodes_from(range(n))
G.add_edges_from([(u, v) for u, v, _ in edge_list])  # drop the weight

# Run classical preprocessing
p = 1
angles, Fp = optimize_angles(G, p=p, n_restarts=5, verbose=True)

print(f"Optimal F_{p}  = {Fp:.6f}")
print(f"Approx ratio  = {approximation_ratio(Fp, G):.4f}")
print(f"γ* = {np.round(angles['gammas'], 4)}")
print(f"β* = {np.round(angles['betas'],  4)}")

KeyboardInterrupt: 